# Extracción de noticias con Media Cloud (API)

Este notebook descarga noticias sobre **Bitcoin/BTC** desde **2025-02-01** hasta **2026-02-01** para la colección **United States – National (#34412234)** y guarda **un parquet por día**.

Salida: `data/news_daily_mediacloud/news_bitcoin_YYYY-MM-DD.parquet`.

In [9]:
from pathlib import Path
import time
import pandas as pd
import mediacloud.api
from datetime import date as Date
import os
from dotenv import load_dotenv

In [ ]:
# Definicion de variables
load_dotenv()
MC_API_KEY = os.getenv("MC_API_KEY") # Key de usuario para la API Media Cloud 
if not MC_API_KEY:
    raise ValueError("No se encontró MC_API_KEY. Revisar que existe en el archivo .env")

COLLECTION_ID = 34412234  # United States - National
QUERY = "(bitcoin OR btc OR Bitcoin OR BTC OR BITCOIN OR Bitcoins OR bitcoins OR Btc)"

START_DATE = "2025-02-01"
END_DATE   = "2026-02-01"

OUT_DIR = Path("data/news_daily_mediacloud")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Reintentos ante errores temporales:
MAX_RETRIES = 8
SLEEP_BETWEEN_REQUESTS = 3
SLEEP_BETWEEN_DAYS = 8


In [ ]:
#Funciones
FINAL_COLS = ["date", "source", "title", "url"]

def safe_get(d, keys, default=None):
    """Prueba varias keys en un dict y devuelve la primera que exista."""
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default

def date_only(x):
    """Devuelve solo la fecha (YYYY-MM-DD) a partir de un valor de fecha/hora."""
    if x is None:
        return None
    ts = pd.to_datetime(x, errors="coerce")
    if pd.isna(ts):
        return None
    return ts.date().isoformat()  # YYYY-MM-DD

def story_to_row(story: dict) -> dict:
    """Mapea una story de MediaCloud al esquema mínimo (date, source, title, url)."""
    url = safe_get(story, ["url", "processed_url", "original_url"])
    title = safe_get(story, ["title", "story_title"], "") or ""
    source = safe_get(story, ["media_name", "publisher", "domain", "source"], "mediacloud")
    created_at = safe_get(story, ["publish_date", "publication_date", "created_at", "indexed_date"])
    return {
        "date": date_only(created_at),    # solo día/mes/año (YYYY-MM-DD)
        "source": str(source),
        "title": str(title),
        "url": str(url) if url is not None else None,
    }

def build_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Limpieza mínima y columnas finales."""
    if df is None or df.empty:
        return pd.DataFrame(columns=FINAL_COLS)

    df = df[FINAL_COLS].copy()

    # Eliminar filas inválidas y duplicados
    df = df.dropna(subset=["url", "date"]).copy()
    df = df.drop_duplicates(subset=["url"]).copy()

    # FILTRO FINAL: solo noticias cuyo TITULO contenga bitcoin/btc/crypto
    TITLE_KEYWORDS_RE = r"(?i)\b(?:bitcoin|btc|crypto)s?\b"
    df = df[df["title"].fillna("").str.contains(TITLE_KEYWORDS_RE, regex=True, na=False)].copy()

    return df.reset_index(drop=True)

def fetch_day(mc: mediacloud.api.SearchApi, day) -> pd.DataFrame:
    """Descarga todas las stories de un día (paginando con pagination_token).
    'day' debe ser datetime.date (si llega str, se convierte a date).
    """
    # Asegurar tipo date
    if isinstance(day, str):
        day = pd.to_datetime(day, errors="raise").date()
    elif isinstance(day, pd.Timestamp):
        day = day.date()
    elif not isinstance(day, Date):
        raise TypeError(f"Day debe ser datetime.date o str YYYY-MM-DD, recibido: {type(day)}")

    all_rows = []
    next_token = None

    while True:
        # Reintentos por si hay errores temporales
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                page, next_token = mc.story_list(
                    QUERY,
                    start_date=day,
                    end_date=day,
                    collection_ids=[COLLECTION_ID],
                    pagination_token=next_token
                )
                break
            except Exception as e:
                msg = str(e)

                # Si parece respuesta vacía o JSON
                if "Expecting value" in msg or "JSON" in msg:
                    # Realizar pausa
                    sleep_s = min(60, 5 * attempt)
                    time.sleep(sleep_s)
                else:
                    time.sleep(1.5 * attempt)

                if attempt >= MAX_RETRIES:
                    raise

        if not page:
            break

        for story in page:
            all_rows.append(story_to_row(story))

        if not next_token:
            break

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    df = pd.DataFrame(all_rows)
    if df.empty:
        return pd.DataFrame(columns=FINAL_COLS)
    return df


In [ ]:
# Descarga dia por dia
mc = mediacloud.api.SearchApi(MC_API_KEY)

start = pd.to_datetime(START_DATE).date()
end   = pd.to_datetime(END_DATE).date()

day = start
total_saved = 0

while day <= end:
    day_str = day.strftime("%Y-%m-%d")
    out_path = OUT_DIR / f"news_bitcoin_{day_str}.parquet"

    if out_path.exists():
        print(f"{day_str}: ya existe {out_path.name}, lo salto")
        day = day + pd.Timedelta(days=1)
        continue

    try:
        df_raw = fetch_day(mc, day_str)
        df_final = build_schema(df_raw)

        df_final.to_parquet(out_path, index=False)
        print(f"📅 {day_str}: guardadas {len(df_final)} noticias → {out_path}")
        total_saved += len(df_final)

    except Exception as e:
        print(f"--> {day_str}: error descargando/guardando: {e}")
        empty = pd.DataFrame(columns=["date","source","title","url"])
        empty.to_parquet(out_path, index=False)
        print(f"!! {day_str}: guardado vacío (para no romper el pipeline) → {out_path}")

    time.sleep(SLEEP_BETWEEN_DAYS)
    day = day + pd.Timedelta(days=1)

print(f"\nTotal noticias guardadas en el rango: {total_saved}")
print(f"Carpeta salida: {OUT_DIR.resolve()}")


In [ ]:
#Comprobacion de los ficheros
files = sorted(OUT_DIR.glob("news_bitcoin_*.parquet"))
print("Parquets generados:", len(files))
if files:
    df_sample = pd.read_parquet(files[-1])
    print("Ejemplo último día:", files[-1].name)
    print(df_sample.head(3))
    print("Columnas:", df_sample.columns.tolist())


In [ ]:
# Limpieza por URL y titular

out_dir = OUT_DIR if isinstance(OUT_DIR, Path) else Path(OUT_DIR)
files = sorted(out_dir.glob("news_bitcoin_*.parquet"))
print(f"Encontrados {len(files)} parquets diarios en: {out_dir}")

def _norm_title(s: pd.Series) -> pd.Series:
    return (
        s.fillna("")
         .astype(str)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
         .str.lower()
    )

total_before = 0
total_after = 0

for fp in files:
    df = pd.read_parquet(fp)
    n_before = len(df)
    total_before += n_before

    # Orden estable para que keep='first' sea reproducible
    sort_cols = []
    if "date" in df.columns:
        sort_cols.append("date")
    sort_cols.append("url")
    df = df.sort_values(sort_cols, kind="mergesort")

    # 1. Deduplicar por URL (mismo artículo repetido)
    df = df.drop_duplicates(subset=["url"], keep="first")

    # 2. Deduplicar por título normalizado (títulos iguales con URL distinta)
    df["_title_norm"] = _norm_title(df["title"])
    df = df.drop_duplicates(subset=["_title_norm"], keep="first").drop(columns=["_title_norm"])

    n_after = len(df)

    # Sobrescribir parquet del día si cambió
    if n_after != n_before:
        df.to_parquet(fp, index=False)

    total_after += n_after
    removed = n_before - n_after

    if removed:
        print(f"{fp.name}: {n_before} -> {n_after} (eliminadas {removed})")
    else:
        print(f"{fp.name}: {n_before} (sin duplicados)")

print(f"\nTotal: {total_before} -> {total_after} (eliminadas {total_before - total_after})")